#Multimodal Car Image Retrieval using CLIP
#Objective

The objective of this notebook is to demonstrate bidirectional multimodal retrieval using the CLIP (Contrastive Language–Image Pretraining) model.

The system is designed to:

* Retrieve images based on a text query (Text → Image)

* Predict text descriptions for a given image (Image → Text)

This is achieved without any task-specific training, using CLIP’s shared image–text embedding space.
#Problem Statement

Traditional image search systems rely on:

* Manual annotations

* Supervised training

* Domain-specific models

These approaches are costly and not scalable.
This notebook explores how CLIP enables semantic understanding between images and text, allowing flexible and scalable multimodal search.
#Model Overview

CLIP is a pretrained multimodal model that learns to associate images and text by training on large-scale image–caption pairs.

Key characteristics:

* Uses dual encoders (image encoder + text encoder)

* Outputs 512-dimensional embeddings

* Supports zero-shot learning

* Works across domains without retraining

#Text → Image Retrieval

This task retrieves the most relevant car images for a given text query.

Workflow:

* Convert input text into an embedding

* Compare against precomputed image embeddings

* Rank images using cosine similarity

* Display top matching images

#Kaggle file upload

In [ ]:
from google.colab import files
files.upload()


In [ ]:
!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [ ]:
!kaggle datasets download -d kshitij192/cars-image-dataset

#Link Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


#Dataset Download and Unzip Dataset

In [ ]:
!kaggle datasets download -d kshitij192/cars-image-dataset


In [ ]:
!unzip /content/cars-image-dataset.zip -d/content/drive/MyDrive/car-images


In [ ]:
!pip install ftfy regex tqdm -q
!pip install git+https://github.com/openai/CLIP.git -q
!pip install faiss-cpu pillow numpy matplotlib -q

## Device Configuration

This cell checks whether a GPU is available and sets the computation device accordingly.

Using a GPU improves the speed of embedding generation, especially for large image datasets.


In [ ]:
import os
import numpy as np
import faiss
import torch
import clip
from PIL import Image
import matplotlib.pyplot as plt

## Loading the CLIP Model

This cell loads the pretrained **CLIP (ViT-B/32)** model.

- The model contains both image and text encoders
- The preprocessing function ensures images are resized and normalized correctly


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

In [ ]:
IMAGE_DIR = "/content/drive/MyDrive/car-images/Cars Dataset/test/Audi"

image_files = sorted([
    f for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith(".jpg")
])

print(f"Images found: {len(image_files)}")
image_files[:5]

## Image Embedding Function

This cell defines a function that:
- Loads an image
- Applies CLIP preprocessing
- Converts the image into a numerical embedding using the CLIP image encoder

The output embedding represents the semantic meaning of the image.


In [ ]:
def embed_image(image_path):
    image = preprocess(Image.open(image_path)).unsqueeze(0).to(device)
    with torch.no_grad():
        embedding = model.encode_image(image)
    return embedding.cpu().numpy().astype("float32")

In [ ]:
def embed_text(text):
    text_tokens = clip.tokenize([text]).to(device)
    with torch.no_grad():
        embedding = model.encode_text(text_tokens)
    return embedding.cpu().numpy().astype("float32")

## Generating Image Embeddings

This cell converts all dataset images into embeddings.

These embeddings are computed once and stored for efficient retrieval during text queries.


In [ ]:
image_embeddings = []
image_paths = []

for img in image_files:
    path = os.path.join(IMAGE_DIR, img)
    emb = embed_image(path)

    image_embeddings.append(emb)
    image_paths.append(path)

image_embeddings = np.vstack(image_embeddings)

print("Embedding shape:", image_embeddings.shape)

## Normalizing Image Embeddings

This cell normalizes all image embeddings using L2 normalization.

Normalization enables the use of **cosine similarity** when comparing text and image embeddings.


In [ ]:
faiss.normalize_L2(image_embeddings)

In [ ]:
embedding_dim = image_embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)
index.add(image_embeddings)

print("Total indexed images:", index.ntotal)

##  Text Embedding Function

This cell defines a function to convert a text query into an embedding using CLIP’s text encoder.

The text embedding lies in the same semantic space as image embeddings.


In [ ]:
query = "An Audi Car "

query_embedding = embed_text(query)
faiss.normalize_L2(query_embedding)

k = 10
scores, indices = index.search(query_embedding, k)

results = [
    (image_paths[i], scores[0][rank])
    for rank, i in enumerate(indices[0])
]

results

In [ ]:
plt.figure(figsize=(15, 4))

for i, (path, score) in enumerate(results):
    img = Image.open(path)
    plt.subplot(1, k, i + 1)
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"Score: {score:.2f}")

plt.show()

## Observations

- Retrieved images closely match the meaning of the text query
- The system works without any task-specific training
- Semantic similarity is captured beyond simple keywords

This confirms the effectiveness of CLIP for text-to-image retrieval.


----------------------------------------

## Objective: Image → Text Retrieval

The objective of this section is to perform **image-to-text retrieval** using the CLIP model.

Given an input car image, the system predicts the **most semantically relevant text description (car brand)** by comparing image and text embeddings.


In [ ]:
import os
import torch
import clip
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


In [ ]:
model, preprocess = clip.load("ViT-B/32", device=device)


In [ ]:
DATASET_DIR = "/content/drive/MyDrive/car-images/Cars Dataset/test"


## Extracting Text Labels from Dataset

This cell extracts folder names from the dataset directory.

Each folder name corresponds to a car brand and is used as a **text label** for image-to-text matching.


In [ ]:
class_names = sorted(os.listdir(DATASET_DIR))
print("Classes:", class_names)


## Creating Natural Language Text Prompts

This cell converts raw class names into meaningful natural language prompts such as
“a photo of a BMW car”.

Using natural language prompts improves CLIP’s semantic understanding.


In [ ]:
text_prompts = [f"a photo of a {cls} car" for cls in class_names]
print(text_prompts)


## Text Embedding Function

This cell defines a function to convert text prompts into numerical embeddings using CLIP’s text encoder.

All embeddings are L2-normalized to enable cosine similarity comparison.


In [ ]:
def embed_texts(texts):
    tokens = clip.tokenize(texts).to(device)
    with torch.no_grad():
        text_embeddings = model.encode_text(tokens)
    # Normalize for cosine similarity
    text_embeddings /= text_embeddings.norm(dim=-1, keepdim=True)
    return text_embeddings


## Generating Text Embeddings

This cell computes embeddings for all text prompts.

These embeddings are generated once and reused during image-to-text prediction for efficiency.


In [ ]:
text_embeddings = embed_texts(text_prompts)


## Image Embedding Function

This cell defines a function to convert an input image into an embedding using CLIP’s image encoder.

The image is preprocessed and normalized before embedding generation.


In [ ]:
def embed_image(image_path):
    image = preprocess(Image.open(image_path)).unsqueeze(0).to(device)
    with torch.no_grad():
        image_embedding = model.encode_image(image)
    # Normalize for cosine similarity
    image_embedding /= image_embedding.norm(dim=-1, keepdim=True)
    return image_embedding


In [ ]:
def predict_car_brand(image_path):
    image_embedding = embed_image(image_path)

    # Cosine similarity
    similarity = (image_embedding @ text_embeddings.T).squeeze(0)

    best_index = similarity.argmax().item()
    best_label = class_names[best_index]
    confidence = similarity[best_index].item()

    return best_label, confidence


In [ ]:
test_image = "/content/drive/MyDrive/car-images/Cars Dataset/test/Audi/1003.jpg"

label, score = predict_car_brand(test_image)

print("Predicted Brand:", label)
print("Similarity Score:", round(score, 3))

plt.imshow(Image.open(test_image))
plt.axis("off")
plt.title(f"Prediction: {label}")


## Summary: Image → Text Retrieval

In this section, an image-to-text retrieval pipeline was implemented using CLIP.

Key highlights:
- Uses dataset folder names as text labels
- Employs natural language prompts for better accuracy
- Relies on cosine similarity in a shared embedding space
- Requires no task-specific training
